In [0]:
%sql
---SQL cell: Shows all databases available in the environment for context and verification.

SHOW DATABASES;

databaseName
default
googleads_bronze
googleads_gold
googleads_silver
information_schema


In [0]:
%sql
---SQL cell: Shows all tables in the 'googleads_gold' database to verify available analytics-ready tables.

SHOW TABLES in googleads_gold;

database,tableName,isTemporary
googleads_gold,dim_ad,false
googleads_gold,dim_ad_group,false
googleads_gold,dim_age_range,false
googleads_gold,dim_asset,false
googleads_gold,dim_campaign,false
googleads_gold,dim_conversion_action,false
googleads_gold,dim_date,false
googleads_gold,dim_device,false
googleads_gold,dim_gender,false
googleads_gold,dim_keyword,false


In [0]:
%sql
-- Creates the dedicated database for Analytics/KPI tables if it doesn't exist.

CREATE DATABASE IF NOT EXISTS googleads_analytics;

-- Verify creation
SHOW DATABASES;

databaseName
default
googleads_analytics
googleads_bronze
googleads_gold
googleads_silver
information_schema


Reminder to use SQL aliases and simple metric names for clarity in queries and dashboards.

In [0]:
%sql
---SQL cell: KPI 1 - High-Level Performance Summary
-- SAVES result to 'kpi_report_1_summary' in the new analytics database

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_1_summary AS
SELECT
    c.campaign_name,
    c.campaign_status,
    
    -- Core Metrics (Renamed for LLM clarity)
    ROUND(SUM(f.cost_inr), 2) AS total_cost_inr,
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    SUM(f.conversions) AS total_leads,
    
    -- Efficiency Metrics
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.clicks), 0), 2) AS average_cpc,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.conversions), 0), 2) AS average_cpl,
    ROUND((SUM(f.clicks) / NULLIF(SUM(f.impressions), 0)) * 100, 2) AS overall_ctr_pct,
    ROUND((SUM(f.conversions) / NULLIF(SUM(f.clicks), 0)) * 100, 2) AS overall_lead_rate_pct

FROM
    googleads_gold.fact_campaign_perf AS f
JOIN
    googleads_gold.dim_campaign AS c 
    ON f.campaign_pk = c.campaign_pk  -- UPDATED: Using Surrogate Key

GROUP BY
    c.campaign_name,
    c.campaign_status
ORDER BY
    total_cost_inr DESC;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_1_summary;

campaign_name,campaign_status,total_cost_inr,total_impressions,total_clicks,total_leads,average_cpc,average_cpl,overall_ctr_pct,overall_lead_rate_pct
Hyderabad,PAUSED,33294.98,244220,10782,4.0,3.09,8323.75,4.41,0.04
Azure Data Engineering -Display - Image,PAUSED,21791.05,691883,66109,0.0,0.33,null,9.55,0.0
Azure Data Engineering -Display - text,PAUSED,20056.12,20040,398,3.0,50.39,6685.37,1.99,0.75
Excel in Data Engineering,PAUSED,18943.69,305164,15280,1.0,1.24,18943.69,5.01,0.01
10-Sept AWS Snowflake,PAUSED,18755.63,53903,444,0.0,42.24,null,0.82,0.0
Azure-Data-Engineering-Jan-2026,ENABLED,16707.35,13257,582,141.0,28.71,118.49,4.39,24.23
Data Engineering (AWS-Sept),PAUSED,6984.65,90137,6905,0.0,1.01,null,7.66,0.0
Leads-Search-24Feb2025,PAUSED,3074.54,5838,170,0.0,18.09,null,2.91,0.0


In [0]:
%sql
---SQL cell: KPI 2 - Campaign Performance Ranking (CPL)
-- SAVES result to 'kpi_report_2_cpl_ranking'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_2_cpl_ranking AS
SELECT
    c.campaign_name,
    c.campaign_status,
    
    -- Metrics
    ROUND(SUM(f.cost_inr), 2) AS total_cost_inr,
    SUM(f.clicks) AS total_clicks,
    SUM(f.conversions) AS total_leads,
    
    -- Calculated KPIs
    ROUND((SUM(f.conversions) / NULLIF(SUM(f.clicks), 0)) * 100, 2) AS lead_rate_pct,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.conversions), 0), 2) AS campaign_cpl

FROM
    googleads_gold.fact_campaign_perf AS f
JOIN
    googleads_gold.dim_campaign AS c 
    ON f.campaign_pk = c.campaign_pk

GROUP BY
    c.campaign_name,
    c.campaign_status

HAVING 
    total_leads > 0  -- Focus purely on campaigns that are generating leads
ORDER BY
    campaign_cpl ASC; -- Rank most efficient (lowest CPL) first

-- Validation
SELECT * FROM googleads_analytics.kpi_report_2_cpl_ranking;

campaign_name,campaign_status,total_cost_inr,total_clicks,total_leads,lead_rate_pct,campaign_cpl
Azure-Data-Engineering-Jan-2026,ENABLED,16707.35,582,141.0,24.23,118.49
Azure Data Engineering -Display - text,PAUSED,20056.12,398,3.0,0.75,6685.37
Hyderabad,PAUSED,33294.98,10782,4.0,0.04,8323.75
Excel in Data Engineering,PAUSED,18943.69,15280,1.0,0.01,18943.69


In [0]:
%sql
---SQL cell: KPI 3 - Top 10 Keywords by Clicks
-- SAVES result to 'kpi_report_3_top_keywords'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_3_top_keywords AS
SELECT
    k.keyword_text,
    k.keyword_match_type,
    
    -- Metrics
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    
    -- Efficiency
    ROUND((SUM(f.clicks) / NULLIF(SUM(f.impressions), 0)) * 100, 2) AS ctr_percentage,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.clicks), 0), 2) AS cost_per_click

FROM
    googleads_gold.fact_keyword_perf AS f
JOIN
    googleads_gold.dim_keyword AS k 
    ON f.keyword_pk = k.keyword_pk  -- UPDATED: Surrogate Key

WHERE
    f.clicks > 0  -- Only interested in keywords driving traffic
GROUP BY
    k.keyword_text,
    k.keyword_match_type
ORDER BY
    total_clicks DESC
LIMIT 25; -- Focus on highest volume drivers

-- Validation
SELECT * FROM googleads_analytics.kpi_report_3_top_keywords;

keyword_text,keyword_match_type,total_impressions,total_clicks,ctr_percentage,cost_per_click
Data Analytics,BROAD,15569,1105,7.1,5.67
it training,BROAD,16330,825,5.05,6.72
software training,BROAD,9738,370,3.8,27.35
Software Training,BROAD,3348,337,10.07,4.19
computer courses near me,BROAD,4404,107,2.43,50.36
Best data engineering courses online,BROAD,8268,100,1.21,46.98
Python programming for beginners,BROAD,19558,96,0.49,7.54
how to become data engineer,BROAD,1625,94,5.78,23.11
online computer courses,BROAD,6845,80,1.17,57.3
data engineer job guarantee course,BROAD,629,71,11.29,32.24


In [0]:
%sql
---SQL cell: KPI 4 - Performance Breakdown by Age Group
-- SAVES result to 'kpi_report_4_age_performance'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_4_age_performance AS
SELECT
    a.age_range,
    
    -- Volume Metrics
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    SUM(f.conversions) AS total_leads,
    
    -- Financial Metrics (Added for context)
    ROUND(SUM(f.cost_inr), 2) AS total_cost_inr,
    
    -- Efficiency Metrics
    ROUND((SUM(f.clicks) / NULLIF(SUM(f.impressions), 0)) * 100, 2) AS ctr_pct,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.clicks), 0), 2) AS cpc_inr,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.conversions), 0), 2) AS cpl_inr

FROM
    googleads_gold.fact_age_perf AS f
JOIN
    googleads_gold.dim_age_range AS a 
    ON f.age_range_pk = a.age_range_pk -- UPDATED: Surrogate Key

GROUP BY
    a.age_range
ORDER BY
    total_leads DESC;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_4_age_performance;

age_range,total_impressions,total_clicks,total_leads,total_cost_inr,ctr_pct,cpc_inr,cpl_inr
18-24,273052,19170,112.0,36259.81,7.02,1.89,323.75
25-34,264602,22917,15.0,25831.66,8.66,1.13,1722.11
35-44,147083,11944,12.0,12029.72,8.12,1.01,1002.48
45-54,55719,4692,5.0,6456.13,8.42,1.38,1291.23
UNDETERMINED,224356,13509,4.0,28471.96,6.02,2.11,7117.99
65-UP,27764,2749,0.0,1871.67,9.9,0.68,null
55-64,36565,3504,0.0,2758.72,9.58,0.79,null


In [0]:
%sql
---SQL cell: KPI 5 - Performance Breakdown by Gender
-- SAVES result to 'kpi_report_5_gender_performance'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_5_gender_performance AS
SELECT
    g.gender,
    
    -- Volume Metrics
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    SUM(f.conversions) AS total_leads,
    
    -- Financial Metrics
    ROUND(SUM(f.cost_inr), 2) AS total_cost_inr,
    
    -- Efficiency Metrics
    ROUND((SUM(f.clicks) / NULLIF(SUM(f.impressions), 0)) * 100, 2) AS ctr_pct,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.clicks), 0), 2) AS cpc_inr,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.conversions), 0), 2) AS cpl_inr

FROM
    googleads_gold.fact_gender_perf AS f
JOIN
    googleads_gold.dim_gender AS g 
    ON f.gender_pk = g.gender_pk -- UPDATED: Surrogate Key

GROUP BY
    g.gender
ORDER BY
    total_leads DESC;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_5_gender_performance;

gender,total_impressions,total_clicks,total_leads,total_cost_inr,ctr_pct,cpc_inr,cpl_inr
MALE,651004,50906,97.0,64256.52,7.82,1.26,662.44
FEMALE,178891,15397,47.0,22120.23,8.61,1.44,470.64
UNDETERMINED,199246,12182,4.0,27302.93,6.11,2.24,6825.73


In [0]:
%sql
---SQL cell: KPI 6 - Monthly Performance Trend
-- SAVES result to 'kpi_report_6_monthly_trend'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_6_monthly_trend AS
SELECT
    d.year,
    d.month,
    
    -- Financials
    ROUND(SUM(f.cost_inr), 2) AS monthly_cost_inr,
    
    -- Conversion Metrics
    SUM(f.conversions) AS monthly_leads,
    
    -- Efficiency
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.conversions), 0), 2) AS monthly_cpl_inr

FROM
    googleads_gold.fact_campaign_perf AS f
JOIN
    googleads_gold.dim_date AS d 
    ON f.date_key = d.date_key

GROUP BY
    d.year,
    d.month
ORDER BY
    d.year,
    d.month;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_6_monthly_trend;

year,month,monthly_cost_inr,monthly_leads,monthly_cpl_inr
2024,7,356.72,0.0,null
2024,8,10366.55,0.0,null
2024,9,6270.74,1.0,6270.74
2024,10,14906.49,2.0,7453.24
2024,11,19133.32,1.0,19133.32
2025,2,11285.81,0.0,null
2025,4,1989.94,0.0,null
2025,5,10616.86,2.0,5308.43
2025,6,9041.25,2.0,4520.62
2025,7,11646.94,0.0,null


In [0]:
%sql
---SQL cell: KPI 7 - Ad Group Efficiency
-- SAVES result to 'kpi_report_7_adgroup_efficiency'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_7_adgroup_efficiency AS
SELECT
    ag.ad_group_name,
    c.campaign_name,
    
    -- Volume Metrics (Aggregated from Keywords)
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    
    -- Financial Metrics
    ROUND(SUM(f.cost_inr), 2) AS total_cost_inr,
    
    -- Efficiency Metrics
    ROUND((SUM(f.clicks) / NULLIF(SUM(f.impressions), 0)) * 100, 2) AS ctr_pct,
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.clicks), 0), 2) AS avg_cpc_inr

FROM
    googleads_gold.fact_keyword_perf AS f
JOIN
    googleads_gold.dim_ad_group AS ag 
    ON f.ad_group_pk = ag.ad_group_pk -- UPDATED: Surrogate Key
JOIN
    googleads_gold.dim_campaign AS c 
    ON f.campaign_pk = c.campaign_pk -- UPDATED: Surrogate Key

GROUP BY
    ag.ad_group_name,
    c.campaign_name
ORDER BY
    ctr_pct DESC;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_7_adgroup_efficiency;

ad_group_name,campaign_name,total_impressions,total_clicks,total_cost_inr,ctr_pct,avg_cpc_inr
Ad group 1,Hyderabad,46000,2459,14184.5,5.35,5.77
Azure Data Engineering – Databricks,Azure-Data-Engineering-Jan-2026,12992,571,16372.23,4.4,28.67
Ad group 1,Leads-Search-24Feb2025,5742,168,3063.09,2.93,18.23
Ad group 1,Azure Data Engineering -Display - text,17371,314,16299.26,1.81,51.91
Academy Of Data Engineering Course Registration Form,10-Sept AWS Snowflake,53549,442,18746.71,0.83,42.41
Ad group 2,Azure Data Engineering -Display - text,3,0,0.0,0.0,null


In [0]:
%sql
---SQL cell: KPI 8 - Performance by Ad Network
-- SAVES result to 'kpi_report_8_network_performance'
-- NOTE: Cost/CPL unavailable at network level in current Bronze/Silver schema. 
-- Switched to Conversion Rate (CVR) for efficiency tracking.

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_8_network_performance AS
SELECT
    n.network_name,
    
    -- Volume Metrics
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    SUM(f.conversions) AS total_leads,
    
    -- Efficiency Metrics
    ROUND((SUM(f.clicks) / NULLIF(SUM(f.impressions), 0)) * 100, 2) AS ctr_pct,
    ROUND((SUM(f.conversions) / NULLIF(SUM(f.clicks), 0)) * 100, 2) AS cvr_pct

FROM
    googleads_gold.fact_campaign_network_perf AS f
JOIN
    googleads_gold.dim_network AS n 
    ON f.network_pk = n.network_pk -- UPDATED: Surrogate Key

GROUP BY
    n.network_name
ORDER BY
    total_leads DESC;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_8_network_performance;

network_name,total_impressions,total_clicks,total_leads,ctr_pct,cvr_pct
SEARCH,39631,1446,146.0,3.65,10.1
SEARCH_PARTNERS,102911,2966,2.0,2.88,0.07
MIXED,395301,22185,1.0,5.61,0.0
CONTENT,886599,74073,0.0,8.35,0.0


In [0]:
%sql
---SQL cell: KPI 9 - Performance by Day of the Week
-- SAVES result to 'kpi_report_9_day_of_week'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_9_day_of_week AS
SELECT
    -- Map integer day to name (1=Sunday in Spark/Java standard)
    CASE d.day_of_week
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_name,
    
    d.day_of_week, -- Keep for sorting
    
    -- Metrics
    SUM(f.impressions) AS total_impressions,
    SUM(f.clicks) AS total_clicks,
    ROUND(SUM(f.cost_inr), 2) AS total_cost_inr,
    SUM(f.conversions) AS total_leads,
    
    -- Efficiency
    ROUND(SUM(f.cost_inr) / NULLIF(SUM(f.conversions), 0), 2) AS cpl_inr

FROM
    googleads_gold.fact_campaign_perf AS f
JOIN
    googleads_gold.dim_date AS d 
    ON f.date_key = d.date_key

GROUP BY
    d.day_of_week,
    day_name
ORDER BY
    d.day_of_week; -- Chronological order (Sun-Sat)

-- Validation
SELECT * FROM googleads_analytics.kpi_report_9_day_of_week;

day_name,day_of_week,total_impressions,total_clicks,total_cost_inr,total_leads,cpl_inr
Sunday,1,145527,11067,15547.6,25.0,621.9
Monday,2,223335,17245,20582.86,18.0,1143.49
Tuesday,3,193451,13169,19596.32,28.0,699.87
Wednesday,4,204521,13887,22123.94,22.0,1005.63
Thursday,5,193946,14171,19715.42,12.0,1642.95
Friday,6,295506,16347,23297.71,20.0,1164.89
Saturday,7,168156,14784,18744.16,24.0,781.01


In [0]:
%sql
---SQL cell: KPI 10 - Ad Type Distribution by Campaign
-- SAVES result to 'kpi_report_10_ad_type_dist'

CREATE OR REPLACE TABLE googleads_analytics.kpi_report_10_ad_type_dist AS
SELECT
    c.campaign_name,
    a.ad_type,
    
    -- Inventory Count
    COUNT(a.ad_pk) AS number_of_ads

FROM
    googleads_gold.dim_ad AS a
JOIN
    googleads_gold.dim_ad_group AS ag 
    ON a.ad_group_id = ag.ad_group_id -- Natural Key Join (Snowflake)
JOIN
    googleads_gold.dim_campaign AS c 
    ON ag.campaign_pk = c.campaign_pk -- Surrogate Key Join

GROUP BY
    c.campaign_name,
    a.ad_type
ORDER BY
    c.campaign_name,
    number_of_ads DESC;

-- Validation
SELECT * FROM googleads_analytics.kpi_report_10_ad_type_dist;

campaign_name,ad_type,number_of_ads
10-Sept AWS Snowflake,RESPONSIVE_SEARCH_AD,1
Azure Data Engineering -Display - Image,RESPONSIVE_DISPLAY_AD,1
Azure Data Engineering -Display - text,CALL_AD,3
Azure Data Engineering -Display - text,RESPONSIVE_SEARCH_AD,1
Azure-Data-Engineering-Jan-2026,RESPONSIVE_SEARCH_AD,1
Hyderabad,RESPONSIVE_SEARCH_AD,1
Leads-Search-24Feb2025,RESPONSIVE_SEARCH_AD,1
